# 03 — KeyDiff Scoring on Flash Attention KV Caches

This notebook validates `keydiff_score`, a standalone implementation
of KeyDiff's key-similarity scoring that works on vLLM-shaped tensors.

KeyDiff scores tokens by how similar their key vectors are to the average
key. Tokens with distinctive keys (dissimilar to the mean) get high scores
and are kept. Tokens with redundant keys (similar to the mean) get low
scores and are candidates for eviction.

We run three experiments:
1. **Standalone scoring** — verify that `keydiff_score` produces correct
   shapes and score distribution
2. **Cross-validation** — verify that our scoring matches kvpress's
   `KeyDiffPress.score()` on the same input
3. **Gather + score pipeline** — scatter keys into a paged cache, gather
   them back, score them, and verify the scores are identical to scoring
   the original dense keys directly

The scoring primitive validated here is used by both compression
strategies (total replacement and append-only filtering) in subsequent
notebooks.

## Imports and Setup

In [ ]:
import torch
import torch.nn.functional as F
from vllm import _custom_ops as ops

torch.manual_seed(42)
torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from kvpress import KeyDiffPress

## Configuration

In [ ]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
SEQ_LEN = 137
DEVICE = "cuda"

## Cache and Gather Functions

Reused from notebook 02 — Flash Attention layout
`[num_blocks, block_size, num_kv_heads, head_size]`.

In [ ]:
def create_kv_caches_flash(num_blocks, block_size, num_kv_heads, head_size,
                           dtype, device="cuda"):
    key_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    return key_cache, value_cache


def build_slot_mapping_for_positions(block_table, positions, block_size):
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping, block_size):
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size
    keys = key_cache[block_indices, offsets]
    values = value_cache[block_indices, offsets]
    return keys, values


print("Cache functions defined")

## Standalone KeyDiff Scoring

KeyDiff scores tokens by how similar their key vectors are to the average
key. Tokens with distinctive keys (dissimilar to the mean) get high scores
and are kept. Tokens with redundant keys (similar to the mean) get low
scores and are candidates for eviction.

The scoring is three operations:
1. L2-normalize all keys
2. Compute the anchor (mean of normalized keys)
3. Score = negative cosine similarity to the anchor

kvpress's `KeyDiffPress.score()` expects keys in shape
`[batch, num_kv_heads, seq_len, head_dim]`. Our standalone version works
on `[seq_len, num_kv_heads, head_dim]` — the shape returned by the Flash
Attention gather — and returns `[num_kv_heads, seq_len]`.

In [ ]:
def keydiff_score(keys):
    """Score keys using KeyDiff's key-similarity metric.

    Parameters
    ----------
    keys : torch.Tensor
        Shape [seq_len, num_kv_heads, head_dim] — the vLLM gather output.

    Returns
    -------
    torch.Tensor
        Shape [num_kv_heads, seq_len]. Higher scores = more important.
    """
    keys_by_head = keys.permute(1, 0, 2)
    normalized = F.normalize(keys_by_head, p=2, dim=-1)
    anchor = normalized.mean(dim=1, keepdim=True)
    scores = -F.cosine_similarity(keys_by_head, anchor, dim=-1)
    return scores


print("keydiff_score defined")

## Experiment 1 — Standalone Scoring

Verify that `keydiff_score` produces correct shapes and score distribution.

In [ ]:
keys = torch.randn(SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)

scores = keydiff_score(keys)

print(f"Keys shape:   {keys.shape}")
print(f"Scores shape: {scores.shape}")
print(f"Score range:  [{scores.min().item():.4f}, {scores.max().item():.4f}]")
print(f"Score mean:   {scores.mean().item():.4f}")
print(f"Score std:    {scores.std().item():.4f}")

## Experiment 2 — Cross-Validation Against kvpress

Verify that our standalone `keydiff_score` produces the same scores as
kvpress's `KeyDiffPress.score()`. The only difference is tensor layout:
vLLM uses `[seq_len, heads, dim]`, kvpress uses `[batch, heads, seq_len, dim]`.

In [ ]:
keys_vllm = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)

our_scores = keydiff_score(keys_vllm)

keys_kvpress = keys_vllm.permute(1, 0, 2).unsqueeze(0)
press = KeyDiffPress()
kvpress_scores = press.score(
    module=None,
    hidden_states=None,
    keys=keys_kvpress,
    values=None,
    attentions=None,
    kwargs={},
)

kvpress_scores_squeezed = kvpress_scores.squeeze(0)

torch.testing.assert_close(
    our_scores, kvpress_scores_squeezed,
    atol=1e-4, rtol=1e-4,
)
print("Scores match kvpress KeyDiffPress.score()")

## Experiment 3 — Gather from Paged Cache + Score

The full pipeline for vLLM integration: keys live in the paged cache,
must be gathered into a dense buffer, then scored. Verify that scoring
gathered keys produces the same result as scoring the original dense keys.

In [ ]:
keys_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)
values_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)

num_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE + 4
key_cache, value_cache = create_kv_caches_flash(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

num_seq_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table = torch.arange(num_seq_blocks, dtype=torch.long, device=DEVICE)
positions = torch.arange(SEQ_LEN, dtype=torch.long, device=DEVICE)
slot_mapping = build_slot_mapping_for_positions(block_table, positions, BLOCK_SIZE)

k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
ops.reshape_and_cache_flash(
    keys_original, values_original,
    key_cache, value_cache,
    slot_mapping, "auto", k_scale, v_scale,
)

print(f"Scattered {SEQ_LEN} tokens into paged cache")

In [ ]:
keys_gathered, _ = gather_from_paged_cache(
    key_cache, value_cache, slot_mapping, BLOCK_SIZE,
)

scores_from_original = keydiff_score(keys_original)
scores_from_gathered = keydiff_score(keys_gathered)

torch.testing.assert_close(
    scores_from_original, scores_from_gathered, atol=0, rtol=0,
)
print("Scoring gathered keys is identical to scoring original keys")

## Notes and Next Steps

**Scoring validated.** `keydiff_score` produces identical results to
kvpress's `KeyDiffPress.score()` and is invariant to the
scatter/gather round trip through the paged cache.

**What this enables:** The scoring primitive is used by both compression
strategies in subsequent notebooks:
- **Notebook 04 — Total replacement:** score all cached tokens, select
  the top-k per head, scatter back into the first `compacted_len` slots
- **Notebook 05 — Append-only filtering:** score cached tokens + new
  token at each decode step, decide whether to keep or skip